In [1]:
import yaml
import pandas as pd
import os
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix)

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

In [2]:
from Titanic.src.features.evaluate_model import evaluate_model, MetricsOrchestrator

In [3]:
# 1. Carregar configurações
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# pipeline selection    
with open(os.path.join("../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# model selection    
with open(os.path.join( "../config/model.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [4]:
# Get feature eng data
pipeline_name = "Pipeline2"
model_name = "rf"

X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)



In [5]:
X_val.drop(
    columns = config_model['single_model']['cols_2_drop'],
    inplace=True
) 

In [6]:
model_path = os.path.join(
        config['init_path'],
        config['single_model']['pkl'],
        f"{model_name}.pkl")
# open model    
with open(model_path, "rb") as file:
    model = pickle.load(file)

In [7]:
pred_proba = model.predict_proba(X_val)[:,1]

In [8]:
def threshold_optimization(pred_proba:np.array, model_name=None, path=None):
    
    dict_tpr = {}
    dict_fpr = {}

    for i in np.linspace(0.01,1,100):
        pred_threshold = np.where(pred_proba >= i,1,0)
        cm = confusion_matrix(y_val, pred_threshold)
        
        tp = cm[1,1] # true positive rate
        tn = cm[0,0] # true negative rate
        fn = cm[1,0] # false positive rate
        fp = cm[0,1] # false positive rate

        dict_tpr[i] = tp / (tp + fn)
        dict_fpr[i] = 1 - (tn / (tn + fp))
        
    np_tpr = np.array(list(dict_tpr.values()))
    np_fpr = np.array(list(dict_fpr.values()))
    np_tpr = np.sort(np_tpr)
    np_fpr = np.sort(np_fpr)

    idx = np.argmax(np_tpr - np_fpr) # max point
    
    print(f'True Positive Rate {np_tpr[idx]}')
    print(f'False Positive rate {np_fpr[idx]}')
    
    df_tpr = pd.DataFrame(
        dict_tpr.values(), 
        index=dict_tpr.keys(), 
        columns=['TPR'])
    df_fpr = pd.DataFrame(
        dict_fpr.values(), 
        index=dict_fpr.keys(), 
        columns=['FPR'])
    df_threshold = df_fpr.join(df_tpr)
    
    df_threshold['diff'] = df_threshold['TPR'] - df_threshold['FPR'] 
    df_threshold.reset_index(inplace=True)
    df_threshold.rename(
        columns={'index':'threshold'}, 
        inplace=True)

    thresh_idx = int(df_threshold['diff'].argmax())
    thresh = df_threshold.iloc[thresh_idx]['threshold']

    print(f'ponto de corte ótimo {thresh}')
    
    if path != None:
        save_path = os.path.join(
            path,
            f"threshold_optimization_{model_name}.png"
    )
         
        plt.figure(figsize=(12,6))
        plt.title('ROC Curve')
        plt.plot(
            df_threshold['FPR'], 
            df_threshold['TPR'], 
            linestyle='--', 
            label = model_name)

        plt.scatter(
            df_threshold.iloc[thresh_idx]['FPR'] ,
            df_threshold.iloc[thresh_idx]['TPR'], 
            color = 'red')

        plt.axhline(
            y = df_threshold.iloc[thresh_idx]['TPR'], 
            linestyle = '--',
            alpha = 0.4, 
            color = 'black')
        plt.axvline(
            x = df_threshold.iloc[thresh_idx]['FPR'], 
            linestyle = '--', 
            alpha = 0.4, 
            color = 'black')

        plt.text(
            x = df_threshold.iloc[thresh_idx]['FPR'], 
            y = df_threshold.iloc[thresh_idx]['TPR'], 
            s = np.round(thresh,3))

        plt.xlabel('1 - specificity (False Positive rate)')
        plt.ylabel('sensitivity (True Positive Rate)')
        plt.legend()
        plt.savefig(
            save_path, 
            dpi=300, 
            bbox_inches="tight"
            )
        plt.close() 
    
    return thresh

In [9]:
threshold = threshold_optimization(
    pred_proba
)

True Positive Rate 0.9324324324324325
False Positive rate 0.23809523809523814
ponto de corte ótimo 0.39


In [14]:
metrics = evaluate_model(model, X_val, y_val)

In [15]:
metrics

{'classification_report': {'0': {'precision': 0.865979381443299,
   'recall': 0.8,
   'f1-score': 0.8316831683168316,
   'support': 105.0},
  '1': {'precision': 0.7439024390243902,
   'recall': 0.8243243243243243,
   'f1-score': 0.782051282051282,
   'support': 74.0},
  'accuracy': 0.8100558659217877,
  'macro avg': {'precision': 0.8049409102338446,
   'recall': 0.8121621621621622,
   'f1-score': 0.8068672251840568,
   'support': 179.0},
  'weighted avg': {'precision': 0.8155118186555936,
   'recall': 0.8100558659217877,
   'f1-score': 0.8111649583523028,
   'support': 179.0}},
 'accuracy_score': 0.8100558659217877,
 'f1_score': 0.782051282051282,
 'roc_auc_score': 0.8121621621621623}

In [13]:
metrics_t

{'classification_report': {'0': {'precision': 0.865979381443299,
   'recall': 0.8,
   'f1-score': 0.8316831683168316,
   'support': 105.0},
  '1': {'precision': 0.7439024390243902,
   'recall': 0.8243243243243243,
   'f1-score': 0.782051282051282,
   'support': 74.0},
  'accuracy': 0.8100558659217877,
  'macro avg': {'precision': 0.8049409102338446,
   'recall': 0.8121621621621622,
   'f1-score': 0.8068672251840568,
   'support': 179.0},
  'weighted avg': {'precision': 0.8155118186555936,
   'recall': 0.8100558659217877,
   'f1-score': 0.8111649583523028,
   'support': 179.0}},
 'accuracy_score': 0.8100558659217877,
 'f1_score': 0.782051282051282,
 'roc_auc_score': 0.8471685971685972}